# 第 1 章 · ROS 2 是什么

本章回答：ROS 明明叫"操作系统"，它到底是什么？一个机器人程序在 ROS 2 里被拆成什么样？为什么要从 ROS 1 推倒重做一个 ROS 2？

配套讲义：仓库里 `ros2/notes/lessons/01-what-is-ros2.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机（macOS）没有装 ROS 2，rclpy 只能在 Linux 上用二进制包装。这里用几十行 Python把总图 2 的关键步骤（匹配、序列化、排队、spin 才叫醒回调）演一遍，数字和讲义一致。

In [ ]:
import json
from collections import deque

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 总图 2 ③ 匹配 | 话题名 + 类型都对上 | `g.publish('/chatter', ...)` | 送到 1 个；拼错成 `/chater` 送到 0 个 |
| 总图 2 ⑥ 序列化 | 往下变字节 | `json.dumps(msg).encode()` | 队列里是 `bytes` |
| 总图 2 ⑨–⑪ | spin 才叫醒回调 | `spin_once(g)` | spin 前 0 条，spin 后 1 条 |
| 总图 2 ⑧ 排队 | 深度 10，满了挤掉最旧的 | `deque(maxlen=10)` | 连发 12 条，收到 3…12 |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**常量 `QUEUE_DEPTH`**

In [ ]:
QUEUE_DEPTH = 10  # 讲义总图 2 第 ⑧ 步：订阅队列深度 10，满了挤掉最旧的

**类 `Subscription`**：一个订阅：话题名、类型、回调，外加一个深度固定的接收队列。

In [ ]:
class Subscription:
    """一个订阅：话题名、类型、回调，外加一个深度固定的接收队列。"""

    def __init__(self, topic: str, msg_type: str, callback, depth: int = QUEUE_DEPTH):
        self.topic = topic
        self.msg_type = msg_type
        self.callback = callback
        self.queue: deque[bytes] = deque(maxlen=depth)  # maxlen：满了自动丢掉最旧的

**类 `Graph`**：计算图的极简版：记下所有订阅；发布时按"话题名 + 类型"匹配（QoS 第 7 章再加）。

In [ ]:
class Graph:
    """计算图的极简版：记下所有订阅；发布时按"话题名 + 类型"匹配（QoS 第 7 章再加）。"""

    def __init__(self):
        self.subscriptions: list[Subscription] = []

    def subscribe(self, topic: str, msg_type: str, callback) -> Subscription:
        sub = Subscription(topic, msg_type, callback)
        self.subscriptions.append(sub)
        return sub

    def publish(self, topic: str, msg_type: str, msg: dict) -> int:
        """⑤–⑧：对象 → 字节（序列化），放进每个匹配上的订阅队列。返回送到了几个订阅。"""
        data = json.dumps(msg).encode()  # 真 ROS 2 用 CDR 格式，这里用 JSON 代替
        n = 0
        for sub in self.subscriptions:
            if sub.topic == topic and sub.msg_type == msg_type:
                sub.queue.append(data)
                n += 1
        return n

**函数 `spin_once`**：⑨–⑪：executor 看哪个队列有数据，取出来、字节 → 对象（反序列化），再调回调。返回调了几次。

In [ ]:
def spin_once(graph: Graph) -> int:
    """⑨–⑪：executor 看哪个队列有数据，取出来、字节 → 对象（反序列化），再调回调。返回调了几次。"""
    calls = 0
    for sub in graph.subscriptions:
        while sub.queue:
            msg = json.loads(sub.queue.popleft())
            sub.callback(msg)
            calls += 1
    return calls

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 · 建立连接：话题名 + 类型都对上才送得到")
g = Graph()
heard: list[str] = []
g.subscribe("/chatter", "std_msgs/msg/String", lambda m: heard.append(m["data"]))
print("  发到 /chatter（String）送到", g.publish("/chatter", "std_msgs/msg/String", {"data": "Hello World: 0"}), "个订阅")
print("  发到 /chater（拼错）送到", g.publish("/chater", "std_msgs/msg/String", {"data": "x"}), "个订阅")

In [ ]:
print("\n第二步 · 往下变字节：队列里存的是 bytes，不是对象")
sub = g.subscriptions[0]
print("  队列里的第一条:", sub.queue[0], " 类型:", type(sub.queue[0]).__name__)

In [ ]:
print("\n第三步 · spin 才叫醒回调")
print("  spin 之前，回调收到", len(heard), "条")
spin_once(g)
print("  spin 之后，回调收到", heard)

In [ ]:
print("\n第四步 · 队列深度 10：不 spin 连发 12 条，最旧的 2 条被挤掉")
heard.clear()
for i in range(1, 13):
    g.publish("/chatter", "std_msgs/msg/String", {"data": f"Hello World: {i}"})
print("  队列里现在有", len(sub.queue), "条")
spin_once(g)
print("  回调收到的第一条:", heard[0], " 最后一条:", heard[-1])

## 3. 代码里才看得到的

1. **本机没有 rclpy**：ROS 2 在 macOS 上不是 Tier 1 平台，只能源码编译。本书的代码层全部用纯 Python 模拟规则，数字和讲义一致；真 API 留到装好 Linux 环境后做里程碑（R-M0 起）。
2. **序列化格式**：模拟用 JSON，真 ROS 2 默认是 CDR（Common Data Representation：通用 / 数据 / 表示，二进制），第 3 章 bag 里看到的 `cdr` 就是它。
3. **`deque(maxlen=10)`** 就是 Keep last 10：Python 的双端队列设了长度上限后，再 append 会自动把最左边（最旧）的挤掉。

## 4. 自己改一改

- 把 `QUEUE_DEPTH` 改成 3，再连发 12 条，看回调收到哪几条；
- 再加一个订阅同一话题的 `Subscription`，看 `publish` 的返回值怎么变（话题是多对多的）。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **架构**：节点写在上，rclpy 来调；rcl 往下是内部，中间件能换插头。
- **流程**：先发现、再匹配；往下变字节，往上变对象；队列里等着，spin 才叫醒回调。
- **四种说话**：流水用话题，问答用服务，长活用动作，设置用参数。
- **为什么 ROS 2**：小饭馆开连锁：去掉领班（master），借用 DDS，加上 QoS。